In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
# pip install pyarrow and dask
import dask.dataframe as dd
import glob

In [2]:
import os 
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [3]:
def assign_lead_time_category(dataframe, variable_name, dataset_name, region_name=''):
  """Takes a dataframe and a variable name, and returns a dataframe with a column
  for the lead time category of the variable.

  Assumptions:
  Desired model has been converted to dataframe format
  You know the variable of interest to compute leads for, such as 'precip'

  The dataset_name argument makes the dataframe more readable and convenient for analysis

  Returns:
  A dataframe with a column for the lead category of the variable.
  """

  # Compute tercile thresholds
  short_lead = 2
  long_lead = 4

  # Classify precipitation using vectorized NumPy operations
  conditions = [
        dataframe[variable_name] < short_lead,
        dataframe[variable_name] >= long_lead
    ]
  choices = ["short", "long"]

  # Assign categories (default is "Medium")
  dataframe[f"lead_category"] = np.select(conditions, choices, default="medium")

  return dataframe

In [ ]:
# Set dictionary of regions and their respective seasons of interest
season_dict = {
    'eastern_east_africa':{
        'MAM': [3,4,5],
        'OND': [10,11,12]
    },
    'lake_victoria_basin':{
        'DJF': [12,1,2],
        'MAM': [3,4,5], 
        'SON': [9,10,11]
    },
    'west_africa':{
        'JAS': [7,8,9]
    },
    'south_sudan':{
        'MJJ': [5,6,7],
        'JAS': [7,8,9],
        'ASO': [8,9,10]
    },
    'eastern_ukraine':{
        'DJF': [12,1,2],
        'AMJ': [4,5,6],
        'JA': [7,8]
    },
    'southern_africa':{
        'DJF': [12,1,2],
        'FMA': [2,3,4]
    },
    'sri_lanka':{
        'ONDJ': [10,11,12,1]
    }
}


# Function to load in data and subset the data into the respective seasons
def convert_monthly_to_seasonal(file_path, seasons_dict, save_path, start_year = 1993, end_year = 2020):
    df_dict = {}
    # extract the model and region from the file path of merged netcdf
    # extract relevant information from the file path
    split = file_path.split('/')
    split[1] = split[1][7:] 
    file_name = split[-1] # get the file name
  
    name_split = file_name.split('_') # get the name of the region, i.e [eastern, east, africa]
    region_name = '_'.join(name_split[0:-2]) # combine the name of the region, i.e eastern_east_africa
    new_file_name= file_name.replace('.nc', '_seasonal.csv') # make new file name for saving
    current_model = name_split[-2]
    # status
    print(f'Converting {file_name} to seasonal...')
    
    # open the merged netcdf, convert to a dataframe, and reset index
    current_netcdf = xr.open_dataset(file_path)
    current_df = current_netcdf.to_dataframe()
    current_df = current_df.reset_index().dropna()
    # take the ensemble mean
    current_df = (current_df
                            .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                            .mean().reset_index())

    # take the spatial means
    current_df = (current_df  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time'])[['predicted_precip', 'precip']]
                              .mean().reset_index())
    
    # seperate month and year into seperate columns
    current_df['month'] = current_df['time'].dt.month
    current_df['year'] = current_df['time'].dt.year
    current_df = current_df.drop(columns=['time'])

    # Subset the df into the respective seasons depending on the region
    for season, months in seasons_dict[region_name].items():
        df = current_df[current_df['month'].isin(months)].copy()
        print(df)
        # calculate correlation between predicted_precip and precip
        df.loc[:, 'season'] = season
        # assign model name column
        df['model'] = current_model
        df['region'] = region_name

        max_lead_time = 6.5
        df = df[df['lead_time'] <= max_lead_time]
        
        # adding date of prediction back
        month_of_prediction = df['month'] - df['lead_time'].astype(int)
        df['year_of_prediction'] = df['year'] - (month_of_prediction < 1).astype(int)  # Adjust year if month < 1
        month_of_prediction[month_of_prediction <= 0] = month_of_prediction[month_of_prediction <= 0] + 12  # Adjust month to be between 1 and 12 
        df['month_of_prediction'] = month_of_prediction

        df = df.rename(columns={'month': 'realization_month', 'year': 'realization_year', 'month_of_prediction': 'month', 'year_of_prediction': 'year'}) # rename columns to match datetime format
        df['date_of_prediction'] = pd.to_datetime(df[['year', 'month']].assign(day=1)) # create date of prediction column
        df = df[(df['realization_year'] >= start_year) & (df['realization_year'] <= end_year) ] # filter to start year
        df = df.drop(columns=['month', 'year', 'realization_month', 'lead_time']) # drop prediction columns

        # seasonal averages
        df = df.groupby(['region', 'model', 'season', 'date_of_prediction', 'realization_year'])[['predicted_precip', 'precip']].mean().reset_index()

        df = df[~(df['date_of_prediction'].dt.month.isin(months[1:]))]

        mode_func = lambda x: x.mode()[0] if not x.mode().empty else np.nan

        df['precip'] = df.groupby(['realization_year', 'season'])['precip'].transform(mode_func)

        df_dict[f"{region_name}_{current_model}_{season}"] = df
        
    final_df = pd.concat(df_dict.values(), ignore_index=True)
    # save as csv
    new_save_path = save_path + '/' + new_file_name
    final_df.to_csv(new_save_path)

    return final_df
        


In [41]:
test_df = convert_monthly_to_seasonal('data/netCDF/eastern_east_africa_CanESM5_merged.nc',
                                      seasons_dict=season_dict,
                                      save_path='data/csv')

Converting eastern_east_africa_CanESM5_merged.nc to seasonal...
      lead_time  predicted_precip     precip  month  year
3           0.5          1.378407  42.901840      3  1991
4           1.5          1.576375  42.901840      3  1991
5           2.5          1.154827  42.901840      3  1991
6           0.5          5.283877  83.088181      4  1991
7           1.5          4.215575  83.088181      4  1991
...         ...               ...        ...    ...   ...
4294        7.5          2.660165  71.152802      5  2021
4295        8.5          2.663025  71.152802      5  2021
4296        9.5          2.701157  71.152802      5  2021
4297       10.5          2.663903  71.152802      5  2021
4298       11.5          2.504623  71.152802      5  2021

[1080 rows x 5 columns]
      lead_time  predicted_precip     precip  month  year
45          0.5          1.685329  30.746590     10  1991
46          1.5          2.266248  30.746590     10  1991
47          2.5          2.027270  30.746

In [91]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
# list_of_files = glob.glob('/content/drive/My Drive/capstone_data/netCDF/*')
list_of_files = glob.glob('data/netCDF/eastern_east_africa*')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = generate_seasons_metrics(f, seasons)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)


In [93]:
# group by lead category and season
final_df = final_df.groupby(['region', 'model', 'season', 'lead_category']).mean().reset_index()
final_df = final_df.dropna()
final_df

,region,model,season,lead_category,corr,potential_skill,conditional_bias,unconditional_bias,skill_score
0,eastern_east_africa,CCSM4,MAM,long,-0.139335,0.022990,0.025949,2.596002,-2.598960
1,eastern_east_africa,CCSM4,MAM,medium,-0.063274,0.004681,0.006232,2.454252,-2.455803
2,eastern_east_africa,CCSM4,MAM,short,0.032988,0.007040,0.006117,2.314680,-2.313757
3,eastern_east_africa,CCSM4,OND,long,0.226491,0.051483,0.045255,2.361025,-2.354796
4,eastern_east_africa,CCSM4,OND,medium,0.168944,0.036738,0.032738,2.371112,-2.367112
...,...,...,...,...,...,...,...,...,...
61,eastern_east_africa,NCEP,MAM,medium,-0.198834,0.041723,0.053545,2.433582,-2.445403
62,eastern_east_africa,NCEP,MAM,short,0.314177,0.099174,0.080627,2.382329,-2.363782
63,eastern_east_africa,NCEP,OND,long,0.418514,0.175154,0.167540,2.437866,-2.430252
64,eastern_east_africa,NCEP,OND,medium,0.108504,0.165758,0.164403,2.426475,-2.425119
